# Round 3 formal Baseline and low-weight KD

This run trains one full supervised Baseline, extracts the InternVL cache, trains one low-weight KD variant, selects thresholds using validation only, and evaluates both selected models on the fixed test split. The fixed test was previously reviewed and is not a blind holdout.

In [ ]:
import json
import os
import subprocess
import sys
import time
from datetime import datetime, timezone
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError('Kaggle GPU is unavailable')
print('GPU:', torch.cuda.get_device_name(0), flush=True)
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
output_dir = Path('/kaggle/working/round3_baseline_kd_outputs')
stage_log_dir = output_dir / 'stage_logs'
stage_log_dir.mkdir(parents=True, exist_ok=True)
stage_status_path = output_dir / 'reports/round2_execution_status.json'
stage_status_path.parent.mkdir(parents=True, exist_ok=True)
stage_records = []

def run_external(stage, command, timeout_seconds, cwd=None):
    record = {'stage': stage, 'status': 'running', 'started_at': datetime.now(timezone.utc).isoformat(), 'timeout_seconds': timeout_seconds}
    stage_records.append(record)
    stage_status_path.write_text(json.dumps(stage_records, ensure_ascii=False, indent=2), encoding='utf-8')
    log_path = stage_log_dir / f'{stage}.log'
    print(f'STAGE START: {stage}', flush=True)
    started = time.perf_counter()
    try:
        with log_path.open('w', encoding='utf-8') as log_file:
            subprocess.run([str(value) for value in command], cwd=cwd, stdout=log_file, stderr=subprocess.STDOUT, timeout=timeout_seconds, check=True)
    except Exception as error:
        record['status'] = 'timeout' if isinstance(error, subprocess.TimeoutExpired) else 'error'
        record['error_type'] = type(error).__name__
        record['exit_code'] = getattr(error, 'returncode', None)
        raise
    else:
        record['status'] = 'complete'
    finally:
        record['finished_at'] = datetime.now(timezone.utc).isoformat()
        record['runtime_seconds'] = time.perf_counter() - started
        record['log_path'] = str(log_path)
        stage_status_path.write_text(json.dumps(stage_records, ensure_ascii=False, indent=2), encoding='utf-8')
        print(f"STAGE END: {stage} status={record['status']} runtime={record['runtime_seconds']:.1f}s", flush=True)


In [ ]:
repo_dir = Path('/kaggle/working/internvl-pv-distillation')
repo_url = 'https://github.com/Zh0416/internvl-pv-distillation.git'
fixed_commit = '974bb9ca7a80ed662aaf257c33e8535801ebb2fb'
run_external('github_clone', ['git', 'clone', repo_url, repo_dir], 300)
run_external('github_checkout', ['git', '-C', repo_dir, 'checkout', '--detach', fixed_commit], 60)
actual_commit = subprocess.check_output(['git', '-C', str(repo_dir), 'rev-parse', 'HEAD'], text=True).strip()
if actual_commit != fixed_commit:
    raise RuntimeError(f'Git commit mismatch: {actual_commit}')
print('GitHub commit:', actual_commit, flush=True)
run_external('install_requirements', [sys.executable, '-m', 'pip', 'install', '--quiet', '-r', repo_dir / 'requirements.txt'], 900)
config_path = repo_dir / 'configs/kaggle_pv_pilot_distill.yaml'


In [ ]:
run_external('prepare_manifest', [sys.executable, '-u', repo_dir / 'scripts/08_prepare_pilot_manifest.py', '--config', config_path, '--output-dir', output_dir], 600, cwd=repo_dir)
manifest = json.loads((output_dir / 'reports/pilot_manifest.json').read_text(encoding='utf-8'))
keys = [row['cache_key'] for rows in manifest['splits'].values() for row in rows]
if len(keys) != 595 or len(set(keys)) != 595:
    raise RuntimeError(f'Unexpected manifest: {len(keys)} total, {len(set(keys))} unique')
print(json.dumps(manifest['counts'], ensure_ascii=False, indent=2), flush=True)


In [ ]:
run_external('teacher_smoke', [sys.executable, '-u', repo_dir / 'scripts/09_extract_pilot_teacher.py', '--config', config_path, '--output-dir', output_dir, '--max-samples', '2'], 1800, cwd=repo_dir)
teacher_report = json.loads((output_dir / 'reports/pilot_teacher_report.json').read_text(encoding='utf-8'))
if teacher_report['success_count'] != 2 or teacher_report['failure_count'] != 0:
    raise RuntimeError(f'Teacher smoke failed: {teacher_report}')
print('Teacher smoke passed:', teacher_report['success_count'], flush=True)


In [ ]:
run_external('baseline_train', [sys.executable, '-u', repo_dir / 'scripts/10_train_pilot_student.py', '--config', config_path, '--output-dir', output_dir, '--variant', 'baseline', '--run-name', 'baseline', '--defer-test'], 3600, cwd=repo_dir)
run_external('baseline_threshold_scan', [sys.executable, '-u', repo_dir / 'scripts/10_train_pilot_student.py', '--config', config_path, '--output-dir', output_dir, '--variant', 'baseline', '--run-name', 'baseline', '--evaluate-only', '--thresholds', '0.3,0.4,0.5,0.6'], 1200, cwd=repo_dir)
baseline_report = json.loads((output_dir / 'reports/pilot_baseline_report.json').read_text(encoding='utf-8'))
print('Baseline validation scan complete:', list(baseline_report['validation_thresholds']), flush=True)


In [ ]:
run_external('teacher_full', [sys.executable, '-u', repo_dir / 'scripts/09_extract_pilot_teacher.py', '--config', config_path, '--output-dir', output_dir], 7200, cwd=repo_dir)
teacher_report = json.loads((output_dir / 'reports/pilot_teacher_report.json').read_text(encoding='utf-8'))
if teacher_report['failure_count'] != 0 or teacher_report['success_count'] != 470:
    raise RuntimeError(f'Full Teacher extraction incomplete: {teacher_report}')
print('Full Teacher cache ready:', teacher_report['success_count'], flush=True)


In [ ]:
def choose_threshold(report):
    evaluations = report['validation_thresholds']
    reference = evaluations['0.5']
    minimum_overall = reference['pixel_metrics']['overall']['dice'] - 0.01
    candidates = [(float(key), value) for key, value in evaluations.items() if value['pixel_metrics']['hard_negative']['false_positive'] == 0 and value['pixel_metrics']['overall']['dice'] >= minimum_overall]
    if not candidates:
        raise RuntimeError('No validation threshold satisfies the hard-negative guard')
    return max(candidates, key=lambda item: (item[1]['pixel_metrics']['hard_positive']['dice'], item[1]['pixel_metrics']['hard_positive']['recall'], item[1]['pixel_metrics']['overall']['dice'], -abs(item[0] - 0.5)))[0]
baseline_threshold = choose_threshold(baseline_report)
run_external('baseline_test', [sys.executable, '-u', repo_dir / 'scripts/10_train_pilot_student.py', '--config', config_path, '--output-dir', output_dir, '--variant', 'baseline', '--run-name', 'baseline', '--evaluate-only', '--thresholds', str(baseline_threshold), '--test'], 1200, cwd=repo_dir)
baseline_report = json.loads((output_dir / 'reports/pilot_baseline_report.json').read_text(encoding='utf-8'))
print('Selected Baseline threshold:', baseline_threshold, flush=True)


In [ ]:
run_external('kd_low_train', [sys.executable, '-u', repo_dir / 'scripts/10_train_pilot_student.py', '--config', config_path, '--output-dir', output_dir, '--variant', 'kd', '--run-name', 'kd_low', '--semantic-kd-weight', '0.1', '--feature-kd-weight', '0.0', '--feature-agreement-only', '--defer-test'], 3600, cwd=repo_dir)
run_external('kd_low_threshold_scan', [sys.executable, '-u', repo_dir / 'scripts/10_train_pilot_student.py', '--config', config_path, '--output-dir', output_dir, '--variant', 'kd', '--run-name', 'kd_low', '--semantic-kd-weight', '0.1', '--feature-kd-weight', '0.0', '--feature-agreement-only', '--evaluate-only', '--thresholds', '0.3,0.4,0.5,0.6'], 1200, cwd=repo_dir)
kd_report = json.loads((output_dir / 'reports/pilot_kd_low_report.json').read_text(encoding='utf-8'))
kd_threshold = choose_threshold(kd_report)
run_external('kd_low_test', [sys.executable, '-u', repo_dir / 'scripts/10_train_pilot_student.py', '--config', config_path, '--output-dir', output_dir, '--variant', 'kd', '--run-name', 'kd_low', '--semantic-kd-weight', '0.1', '--feature-kd-weight', '0.0', '--feature-agreement-only', '--evaluate-only', '--thresholds', str(kd_threshold), '--test'], 1200, cwd=repo_dir)
kd_report = json.loads((output_dir / 'reports/pilot_kd_low_report.json').read_text(encoding='utf-8'))
baseline_metrics = baseline_report['test']['pixel_metrics']
kd_metrics = kd_report['test']['pixel_metrics']
comparison = {'baseline_threshold': baseline_threshold, 'kd_threshold': kd_threshold, 'baseline': baseline_metrics, 'kd': kd_metrics, 'delta': {group: {metric: kd_metrics[group][metric] - baseline_metrics[group][metric] for metric in ('dice', 'iou', 'precision', 'recall', 'specificity')} for group in ('overall', 'regular_positive', 'hard_positive', 'hard_negative')}}
(output_dir / 'reports/round3_comparison.json').write_text(json.dumps(comparison, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps({'kd_threshold': kd_threshold, 'comparison': comparison['delta']}, ensure_ascii=False, indent=2), flush=True)


In [ ]:
status = json.loads(stage_status_path.read_text(encoding='utf-8'))
if any(item['status'] != 'complete' for item in status):
    raise RuntimeError(f'Stage failure: {status}')
print(json.dumps({'round3_completed': True, 'stages': status, 'output_dir': str(output_dir)}, ensure_ascii=False, indent=2), flush=True)
